# General information
<span style="color: green"> **Please use the BIDS structure**</span> so that pipeline can run smoothly. Place ipynb-notebooks on the top level of the project folder (same level as subject folders).


**Gereral processing steps:**  

0. [Import libraries](#0-import-libraries)  
1. [Finding all EEG recording](#1-list-all-eeg-recordings-and-log-files)  
2. [Correcting channel information](#2-correct-the-channel-names-in-vhdr-file)  
3. [Processing the experimental log files](#3-process-experimental-log)  
4. [Setting new annotations at direction cue and initial ground contact](#4-match-eeg-and-force-plate-data)  
5. [Concatenating multiple recordings of one session](#5-concatenate-block-recordings)  

::: {#eeg-process}

![](Fig2-EEGProcess.svg){width=80%}

EEG processing overview

:::

# 0. Import libraries

The analysis pipeline is based on the following libraries. In case of an error in the execution of this cell, probably one or more of the libraries is not installed. In this case, start a terminal in **ANACONDA.NAVIGATOR** *Environments>Terminal* and install the library in question using the command <span style='color: red'>*pip install [library name]*</span>.

In [1]:
import warnings                 # switch of pandas and mne warnings
warnings.simplefilter(action='ignore')

import requests
import os
import os.path as op
import time
from pathlib import Path

import mne

import numpy as np
import pandas as pd
import ipyfilechooser
import ipywidgets as widgets
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import pyprep
from pyprep.prep_pipeline import PrepPipeline
from datetime import datetime, timedelta
from datetime import datetime, timezone
#from playsound import playsound
from openpyxl import load_workbook
from mne.export import export_raw

from collections import defaultdict
from mne.datasets import fetch_fsaverage
from mne.evoked import combine_evoked
from mne.forward import make_forward_dipole
from mne.simulation import simulate_evoked
from mne.viz import circular_layout
from mne_connectivity import spectral_connectivity_epochs
from mne_connectivity.viz import plot_connectivity_circle
from mne_icalabel import label_components
from mne_icalabel.gui import label_ica_components
from asrpy import ASR

from nilearn.image import index_img
from nilearn.plotting import plot_stat_map
from mne import read_evokeds
from mne.datasets import sample
from mne.minimum_norm import  make_inverse_operator, apply_inverse, read_inverse_operator, apply_inverse_epochs

%matplotlib qt

mne.set_log_level("ERROR")  # only show errors (no warnings or information messages)

# 1. List all EEG recordings and Log-files
Prepare data to match EEG and force plate data


In [2]:
# -------------------------------------------------------------------------
# PATH SETUP
# Define the project root as the parent of this script's location.
# pathlib.Path is used for cross-platform compatibility (Linux, Windows, macOS).
# If running in a Jupyter notebook, replace Path(__file__).parent with
# Path.cwd() or set the project root explicitly as Path("/your/project/path").
# -------------------------------------------------------------------------
wd = Path.cwd()  # project root; code lives at the same level as subject folders

# -------------------------------------------------------------------------
# SUBJECT DISCOVERY
# Identify all subject folders following BIDS convention (sub-XX).
# Folders not matching this prefix are ignored.
# -------------------------------------------------------------------------
sub_folders = sorted([f for f in wd.iterdir()
                      if f.is_dir() and f.name.startswith("sub-")])

# -------------------------------------------------------------------------
# FILE COLLECTION
# For each subject and session, collect:
#   eD : EEG header files (.vhdr, BrainVision format)
#   lD : behavioral log files (.txt) from the beh/ folder
# Files prefixed with "._" are macOS metadata artifacts and are skipped.
# -------------------------------------------------------------------------
eD = []  # EEG header files (.vhdr)
lD = []  # behavioral log files (.txt)

for sub in sub_folders:
    # iterate over session folders (ses-XX) within each subject
    for ses in sorted([f for f in sub.iterdir()
                       if f.is_dir() and f.name.startswith("ses-")]):

        # --- EEG data ---
        eeg_path = ses / "eeg"
        if eeg_path.is_dir():
            for f in sorted(eeg_path.iterdir()):
                if f.name.startswith("._"):  # skip macOS metadata
                    continue
                if f.suffix == ".vhdr":
                    eD.append(f)

        # --- Behavioral log files ---
        beh_path = ses / "beh"
        if beh_path.is_dir():
            for f in sorted(beh_path.iterdir()):
                if f.name.startswith("._"):  # skip macOS metadata
                    continue
                if f.suffix == ".txt":
                    lD.append(f)

# -------------------------------------------------------------------------
# SUMMARY
# Print the number of files found per modality for quick sanity check.
# -------------------------------------------------------------------------
print(f"EEG files found   : {len(eD)}")
print(f"Log files found   : {len(lD)}")

EEG files found   : 7
Log files found   : 2


# 2. Correct the channel names in vhdr-file
The original .vhdr file needs to be edited since the *Iz* channel is incorrectly named *FCz*.

For correct montage: Change FCz to Iz in every vhdr-file (all files in eD)

> *Iz* -> *FCz* in .vhdr file

In [3]:
# -------------------------------------------------------------------------
# CHANNEL LABEL CORRECTION
# In the BrainVision .vhdr files, the "Iz" channel was incorrectly labeled
# as "FCz". This block corrects it to "Iz" across all subject files.
# Expected: exactly 3 occurrences of "FCz" per file (channel infos,
# channels, and impedances.
# Note: str.replace() and str.count() are case-sensitive in Python,
# so "Iz" will not accidentally match substrings like "size".
# -------------------------------------------------------------------------

for fp in eD:
    with open(fp, "r", encoding="utf-8") as f:
        content = f.read()

    count_fcz = content.count("FCz")

    # --- Already corrected ---
    # If FCz is not found, the file may have been processed in a previous run.
    # We confirm by checking whether Iz is already present as expected.
    if count_fcz == 0:
        if content.count("Iz") >= 3:
            print(f"Already corrected, skipping : {fp.name}")
        else:
            print(f"Warning — FCz not found but Iz also missing, manual check needed : {fp.name}")
        continue

    # --- Unexpected number of occurrences ---
    # Exactly 3 occurrences are expected. Any other count suggests a file
    # structure change and should be flagged for manual inspection.
    if count_fcz != 3:
        print(f"Warning — {count_fcz} occurrences of 'FCz' found (expected 3), skipping : {fp.name}")
        continue

    # --- Replace and write ---
    content_new = content.replace("FCz", "Iz")

    with open(fp, "w", encoding="utf-8") as f:
        f.write(content_new)

    #print(f"Replaced 'FCz' → 'Iz' ({count_fcz} occurrences) : {fp.name}")

print(f"\nChannel label correction complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Already corrected, skipping : sub-01_ses-01_task-sidecut_run-01_eeg.vhdr
Already corrected, skipping : sub-01_ses-01_task-sidecut_run-02_eeg.vhdr
Already corrected, skipping : sub-01_ses-01_task-sidecut_run-03_eeg.vhdr
Already corrected, skipping : sub-01_ses-01_task-sidecut_run-04_eeg.vhdr
Already corrected, skipping : sub-02_ses-01_task-sidecut_run-01_eeg.vhdr
Already corrected, skipping : sub-02_ses-01_task-sidecut_run-02_eeg.vhdr
Already corrected, skipping : sub-02_ses-01_task-sidecut_run-03_eeg.vhdr

Channel label correction complete: 2026-05-16 15:10:18


# 3. Process experimental log

 Write clean experimental Log. 
 Track amount of correct trials (== # of QTM files)

## 3.1. Find initial ground contact (IC) in force files

In [4]:
# -------------------------------------------------------------------------
# CLASS: forceData
# Represents the vertical ground reaction force (vGRF) data from a single
# force plate recording file (tab-delimited .txt).
# On instantiation, the file is loaded and the time of initial contact (TOI)
# is detected automatically.
#
# Parameters
# ----------
# file      : str or Path — path to the force plate .txt file
# threshold : float — force threshold in Newtons to detect initial contact
# -------------------------------------------------------------------------
class forceData:

    def __init__(self, file, threshold):
        self.File = Path(file)              # path to the source file (Path object for cross-platform compatibility)
        self.Start = ""                     # recording start time (hh:mm:ss), read from file header
        self.DirectionCue = -1              # direction cue (e.g. leg used); to be set externally if needed
        self.Frequency = -1                 # sampling frequency in Hz, read from file header
        self.Time = -1                      # list of time stamps in ms
        self.Force = -1                     # list of offset-corrected, inverted vGRF values in N
        self.TOI = -1                       # time of initial contact in ms (threshold crossing)
        self.TOIthresholdN = threshold      # force threshold in N for TOI detection

        self.load()
        self.findTOI()

    # -------------------------------------------------------------------------
    # METHOD: load
    # Reads the force plate file and populates Frequency, Start, Time, and Force.
    # File structure:
    #   Row 2  : sampling frequency
    #   Row 3  : recording start timestamp
    #   Row 28+: time series data (col 1 = time, col 4 = vertical force)
    # The first 100 samples are used to estimate and subtract the static offset.
    # The z-axis is inverted (* -1) to yield positive values for ground contact.
    # -------------------------------------------------------------------------
    def load(self):
        # Read sampling frequency from second row, second column
        self.Frequency = pd.read_table(
            self.File, delimiter="\t", skiprows=1, nrows=1, header=None
        )[1][0]

        # Read start timestamp from third row (format: hh:mm:ss), strip trailing characters
        self.Start = pd.read_table(
            self.File, delimiter="\t", skiprows=2, nrows=1, header=None
        )[1][0].split(",")[1].lstrip()[0:-4]

        # Read time series data starting at row 28
        d = pd.read_table(self.File, delimiter="\t", skiprows=27, header=None)

        # Store time vector (column 1)
        self.Time = d[1].to_list()

        # Compute static offset as mean of first 100 samples (pre-contact baseline)
        offset = d[4].iloc[:100].mean()

        # Subtract offset and invert z-axis so contact force is positive
        self.Force = [(val - offset) * -1 for val in d[4].to_list()]

    # -------------------------------------------------------------------------
    # METHOD: findTOI
    # Detects the time of initial contact (TOI) as the first time point where
    # vGRF exceeds the defined threshold. TOI remains -1 if threshold is never
    # exceeded (e.g. missing contact or corrupted trial).
    # -------------------------------------------------------------------------
    def findTOI(self):
        force_array = np.array(self.Force)

        if force_array.max() > self.TOIthresholdN:
            # Find first index where force exceeds threshold
            first_crossing = np.argwhere(force_array > self.TOIthresholdN)[0][0]
            self.TOI = self.Time[first_crossing]

    # -------------------------------------------------------------------------
    # METHOD: plot
    # Generates a time-series plot of the vGRF with reference lines for:
    #   - zero baseline
    #   - force threshold (dotted)
    #   - TOI (dashed vertical line)
    # -------------------------------------------------------------------------
    def plot(self):
        fig, ax = plt.subplots()

        # Plot vGRF time series
        ax.plot(self.Time, self.Force, label="vGRF [N]")

        # Zero baseline
        ax.axhline(y=0, color="k", linewidth=0.5)

        # Force threshold reference line
        ax.axhline(y=self.TOIthresholdN, color="k", linestyle=":", linewidth=0.5)

        # TOI vertical marker
        if self.TOI != -1:
            ax.axvline(
                x=self.TOI,
                color="k",
                linestyle="--",
                linewidth=0.5,
                label=f"TOI = {self.TOI} ms"
            )

        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Force (N)")
        ax.set_title(self.File.name)  # show only filename, not full path
        ax.legend()
        plt.show()

In [ ]:
# -------------------------------------------------------------------------
# FUNCTION: getExpLog
# Loads and processes the experimental log file for a single subject/session.
# Matches behavioral log entries to motion capture (QTM) files via timestamp,
# extracts the time of initial contact (TOI) from force plate data, and
# computes the anticipatory time to response (ATR = TOI - RS).
#
# Parameters
# ----------
# path     : str or Path — subject/session root directory
# fileName : str or Path — path to the behavioral log .txt file
#
# Returns
# -------
# expLog   : pd.DataFrame — processed trial log with TOI, ATR, and session info
# files    : list of Path — list of matched QTM marker .tsv files
# -------------------------------------------------------------------------
def getExpLog(path, fileName):
    path = Path(path)
    fileName = Path(fileName)

    # -------------------------------------------------------------------------
    # LOAD AND CLEAN BEHAVIORAL LOG
    # The first row contains only program startup info and is discarded.
    # Columns are renamed to consistent internal labels.
    # -------------------------------------------------------------------------
    expLog = pd.read_table(fileName, delimiter=" ")

    # Drop first row (program startup entry) and reset index
    expLog = expLog.iloc[1:].reset_index(drop=True)

    # Select relevant columns and rename for clarity
    expLog = expLog[["HOn", "RS", "Jump_No", "Filename", "Status"]]
    expLog = expLog.rename(columns={
        "Jump_No": "Arrow_Direction",
        "Filename": "Forcefile"
    })

    # HOn contains additional metadata after a comma — keep only the timestamp
    expLog["HOn"] = expLog["HOn"].str.split(",").str[0]

    # Initialize columns to be filled during file matching below
    expLog["QTM_filename"] = ""     # path to matched QTM marker file
    expLog["RScorrect"] = -1.0      # reaction stimulus timepoint from QTM (ms)
    expLog["IC"] = -1.0             # initial contact timepoint from force data (ms)
    expLog["ATR"] = -1.0            # anticipatory time to response: TOI - RS (ms)
    expLog["Run"] = 0               # Run (block) number, extracted from folder name

    # -------------------------------------------------------------------------
    # DISCOVER QTM MARKER FILES
    # Motion capture data is stored in "sub-XX/ses-XX/motion", organized in block subfolders (SCun1 to SCun4).
    # Only folders containing "SCun" (sidecut maneuver) are included.
    # Only .tsv files without underscores in the name are marker data files
    # (files with underscores are force plate files: _f_1.tsv, _f_2.tsv).
    # -------------------------------------------------------------------------
    kmp_path = path / "motion"  
    files = []

    for trial_dir in sorted(kmp_path.iterdir()):
        if "SCun" in trial_dir.name and trial_dir.is_dir():
            for f in sorted(trial_dir.iterdir()):
                # Exclude force plate files (contain underscore) and non-tsv files
                if "_" not in f.name and f.suffix == ".tsv":
                    files.append(f)
    
    print(f"Found {len(files)} QTM marker files")

    # -------------------------------------------------------------------------
    # MATCH QTM FILES TO LOG ENTRIES VIA TIMESTAMP
    # Each QTM marker file contains a timestamp in its header (row 7).
    # This is matched against the HOn column in the behavioral log.
    # When a match is found:
    #   - The QTM filename is recorded
    #   - The reaction stimulus (RS) timepoint is read from the marker file
    #   - The force plate file is selected based on sidecut direction:
    #       Left  → _f_2.tsv (right force plate)
    #       Right → _f_1.tsv (left force plate)
    #   - TOI is extracted via the forceData class
    #   - ATR is computed as TOI - RS
    # -------------------------------------------------------------------------
    for qtm_file in files:
        # Read timestamp from header (row 7, DESCRIPTION column)
        ts = pd.read_table(qtm_file, delimiter="\t", skiprows=6, nrows=1)
        t = ts["DESCRIPTION"].iloc[0].split(", ")[1].split(".")[0]

        # Read reaction stimulus timepoint from row 9 (3D column)
        rs = pd.read_table(qtm_file, delimiter="\t", skiprows=8, nrows=1)

        # Match against each log entry by timestamp
        for jj in range(len(expLog)):
            if expLog["HOn"][jj] != t:
                continue

            # Record matched QTM file
            expLog.at[jj, "QTM_filename"] = str(qtm_file.relative_to(wd))

            # Extract RS timepoint from marker file
            expLog.at[jj, "RScorrect"] = rs["3D"].iloc[0]

            # Select force plate file based on direction cue
            # Left sidecut → right force plate (_f_2), Right sidecut → left (_f_1)
            force_suffix = "_f_2.tsv" if expLog["Arrow_Direction"][jj] == "Left" else "_f_1.tsv"
            force_file = qtm_file.with_name(qtm_file.stem + force_suffix)

            # Load force data and extract TOI
            fd = forceData(force_file, threshold=20)
            expLog.at[jj, "IC"] = fd.TOI

            # ATR: anticipatory time to response (ms between RS and initial contact)
            expLog.at[jj, "ATR"] = fd.TOI - rs["3D"].iloc[0]

    # -------------------------------------------------------------------------
    # ASSIGN SESSION NUMBERS
    # Session number is encoded as the last character of the parent folder name
    # (e.g. "SCun_S1" → session 1). Only rows with a matched QTM file are updated.
    # -------------------------------------------------------------------------
    for ii in range(len(expLog)):
        if len(expLog["QTM_filename"][ii]) > 0:
            qtm_path = Path(expLog["QTM_filename"][ii])
            expLog.at[ii, "Run"] = int(qtm_path.parent.name.split("_")[-1])

    # -------------------------------------------------------------------------
    # FILTER INVALID TRIALS
    # Remove trials marked as incorrect (Status == 0) and trials where
    # initial contact was not detected (IC <= 0, i.e. TOI remained at -1).
    # -------------------------------------------------------------------------
    expLog = expLog[expLog["Status"] != 0]
    expLog = expLog[expLog["IC"] > 0]
    expLog.index = np.arange(len(expLog))

    return expLog, files

In [6]:
# -------------------------------------------------------------------------
# CONFIGURATION
# Define which sessions to process. Adjust as needed.
# -------------------------------------------------------------------------
SESSIONS_TO_PROCESS = None  # e.g. ["ses-01", "ses-02"] or None to process all sessions

derivative_type = "annot"
DERIVATIVES_PATH = wd / "derivatives" / f"01_{derivative_type}"  # output root for processed logs

# -------------------------------------------------------------------------
# PROCESS EXPERIMENTAL LOGS AND SAVE
# lD contains full paths to behavioral log files (sub-XX/ses-XX/beh/*.txt).
# Session and subject paths are derived from the log file path.
# Processed expLog is saved to:
#   derivatives/01_annot/sub-XX/ses-XX/beh/<filename>.xlsx
# -------------------------------------------------------------------------

for log_file_path in lD:
    log_file_path = Path(log_file_path)

    # Derive session and subject from path structure
    ses_path = log_file_path.parent.parent   # beh/ → ses-XX
    sub_path = ses_path.parent               # ses-XX → sub-XX

    # Skip sessions not in the configured list (if filter is active)
    if SESSIONS_TO_PROCESS is not None and ses_path.name not in SESSIONS_TO_PROCESS:
        print(f"Skipping : {sub_path.name} / {ses_path.name}")
        continue

    try:
        # Process log file — match QTM/force data, compute TOI and ATR
        expLog, files = getExpLog(ses_path, log_file_path)

        # Construct output path under derivatives/01_annot/
        output_dir = DERIVATIVES_PATH / sub_path.name / ses_path.name / "beh"
        output_dir.mkdir(parents=True, exist_ok=True)  # create folder if it doesn't exist

        output_path = output_dir / log_file_path.with_suffix(".xlsx").name
        expLog.to_excel(output_path, index=False)

        print(f"Saved : derivatives / {derivative_type} / {sub_path.name} / {ses_path.name} / beh → {output_path.name}")

    except Exception as e:
        print(f"Error processing {sub_path.name} / {ses_path.name} : {e}")

Found 124 QTM marker files
Saved : derivatives / annot / sub-01 / ses-01 / beh → 20240710_112742_log.xlsx
Found 136 QTM marker files
Saved : derivatives / annot / sub-02 / ses-01 / beh → 20240703_131728_log.xlsx


## 3.2. Summary of trial counts 
summary of QTM files (= # of correctly executed trials) for left and right trials


In [7]:
# -------------------------------------------------------------------------
# TRIAL LOG SUMMARY
# Scans derivatives/01_annot/ for processed experiment logs (*_log.xlsx),
# counts valid trials per direction and session block, and writes one summary
# Excel file per session into the corresponding ses-XX/ folder.
# -------------------------------------------------------------------------

DERIVATIVES_PATH = wd / "derivatives" / "01_annot"

# -------------------------------------------------------------------------
# DISCOVER PROCESSED LOG FILES
# Files are expected at: derivatives/01_annot/sub-XX/ses-XX/beh/*_log.xlsx
# -------------------------------------------------------------------------
log_list = sorted([
    f for f in DERIVATIVES_PATH.rglob("*_log.xlsx")
    if not f.name.startswith("._")
])

print(f"Found {len(log_list)} processed log files")

# -------------------------------------------------------------------------
# GROUP LOG FILES BY SESSION
# Keys are session folder names (e.g. "ses-01").
# -------------------------------------------------------------------------
from collections import defaultdict
session_logs = defaultdict(list)

for lf in log_list:
    ses_name = lf.parent.parent.name  # beh/ → ses-XX
    session_logs[ses_name].append(lf)

# -------------------------------------------------------------------------
# COUNT TRIALS PER SUBJECT AND SESSION
# One summary DataFrame per session, saved at ses-XX/ level across all subjects.
# NOTE: Arrow Direction refers to the direction cue shown to the participant.
# Arrow Direction == "Left" -> sidecut with right leg -> logged as "QTM Right"
# Arrow Direction == "Right" -> sidecut with left leg -> logged as "QTM Left"
# -------------------------------------------------------------------------
for ses_name, lf_list in sorted(session_logs.items()):

    sum_data = {
        "File Name"        : [],
        "ID"               : [],
        "QTM Left block 1" : [],
        "QTM Right block 1": [],
        "QTM Left block 2" : [],
        "QTM Right block 2": [],
        "QTM Left block 3" : [],
        "QTM Right block 3": [],
        "QTM Left block 4" : [],
        "QTM Right block 4": [],
        "QTM Left total"   : [],
        "QTM Right total"  : [],
        "QTM total"        : [],
    }

    for lf in sorted(lf_list):
        try:
            eL = pd.read_excel(lf)

            # Subject ID from grandparent folder (sub-XX)
            ID = lf.parent.parent.parent.name  # beh/ → ses-XX → sub-XX

            # Count trials per session block and direction
            counts = {}
            for sess in [1, 2, 3, 4]:
                counts[f"left_{sess}"]  = ((eL["Run"] == sess) & (eL["Arrow_Direction"] == "Right")).sum()
                counts[f"right_{sess}"] = ((eL["Run"] == sess) & (eL["Arrow_Direction"] == "Left")).sum()

            total_left  = (eL["Arrow_Direction"] == "Right").sum()
            total_right = (eL["Arrow_Direction"] == "Left").sum()
            total       = len(eL)

            sum_data["File Name"].append(str(lf))
            sum_data["ID"].append(ID)
            for sess in [1, 2, 3, 4]:
                sum_data[f"QTM Left block {sess}"].append(counts[f"left_{sess}"])
                sum_data[f"QTM Right block {sess}"].append(counts[f"right_{sess}"])
            sum_data["QTM Left total"].append(total_left)
            sum_data["QTM Right total"].append(total_right)
            sum_data["QTM total"].append(total)

            print(f"Processed : {ID} / {ses_name} ({total} trials)")

        except Exception as e:
            print(f"Error processing {lf.name} : {e}")

    # -------------------------------------------------------------------------
    # WRITE SUMMARY TO SESSION FOLDER
    # Saved at: derivatives/01_annot/ses-XX/YYYYMMDD_trialslog_summary.xlsx
    # -------------------------------------------------------------------------
    sum_df = pd.DataFrame(sum_data)

    date = datetime.now().strftime("%Y%m%d")
    output_path = DERIVATIVES_PATH / f"trialslogSummary_{ses_name}_{date}.xlsx"

    with pd.ExcelWriter(output_path, mode="w", engine="openpyxl") as writer:
        sum_df.to_excel(writer, sheet_name="QTM Trials", index=False)

    print(f"\nSummary saved : {output_path}")
    print(f"Subjects in {ses_name} : {len(sum_df)}\n")

Found 2 processed log files
Processed : sub-01 / ses-01 (123 trials)
Processed : sub-02 / ses-01 (135 trials)

Summary saved : /home/joel/Documents/BIDS - Kopie/CoMoCut_Proof-of-conept/derivatives/01_annot/trialslogSummary_ses-01_20260516.xlsx
Subjects in ses-01 : 2



# 4. Match EEG and force plate data

## 4.1. Set events

Set events at time of:
- direction signal (RS_left, RS_right)
- initial ground contact (IC_left, IC_right)

Save events.csv / events.tsv file

In [8]:
# -------------------------------------------------------------------------
# FUNCTION: getTimeDiff
# Computes the absolute time difference between a direction cue (RS) timestamp
# (string from expLog, format hh:mm:ss,xx) and an EEG trigger timestamp
# (datetime object from MNE annotations).
# The date is taken from the EEG trigger since the log only stores time.
#
# Parameters
# ----------
# eL     : str — RS timestamp string from expLog (e.g. "11:47:26,93")
# trigTS : datetime — EEG trigger timestamp from MNE annotation
#
# Returns
# -------
# dT : timedelta — absolute time difference between the two timestamps
# -------------------------------------------------------------------------
def getTimeDiff(eL, trigTS):
    # Parse time components from the direction cue timestamp string
    h   = int(eL.split(":")[0])
    min = int(eL.split(":")[1])
    s   = int(eL.split(":")[2].split(",")[0])
    us  = int(eL.split(":")[2].split(",")[1]) * 100

    # Reconstruct as datetime using the EEG trigger date as reference
    rs = datetime(
        trigTS.year, trigTS.month, trigTS.day,
        h, min, s, us,
        tzinfo=timezone.utc
    )

    return abs(trigTS - rs)

In [9]:
# -------------------------------------------------------------------------
# FUNCTION: setAnnotations
# Matches EEG triggers (S14 = direction cue) to expLog entries via
# timestamp proximity, then sets two annotations per matched trial:
#   RS_<leg>  : onset of the rdirection cue
#   IC_<leg>  : onset of initial contact (RS onset + ATR)
#
# Erroneous duplicate S14 triggers within 8 seconds of each other are
# suppressed (only the first is kept).
#
# Parameters
# ----------
# data : mne.io.Raw — loaded EEG data with original annotations
# eL   : pd.DataFrame or str/Path — expLog DataFrame or path to .xlsx
#
# Returns
# -------
# dataAnno : mne.io.Raw — EEG data with updated annotations
# -------------------------------------------------------------------------
def setAnnotations(data, eL):
    # Accept either a DataFrame or a path to an Excel file
    if not isinstance(eL, pd.DataFrame):
        eL = pd.read_excel(eL)

    ons   = []  # annotation onsets (s, relative to recording start)
    dur   = []  # annotation durations (s)
    descr = []  # annotation labels

    lastS14 = None  # timestamp of previous valid S14 trigger

    for ii in data.annotations:
        if ii["description"] != "Stimulus/S 14":
            continue

        # Absolute timestamp of this S14 trigger
        curS14 = ii["orig_time"] + timedelta(seconds=np.round(ii["onset"]))

        # Suppress duplicate triggers within 8 s of the previous valid one
        if lastS14 is not None and (curS14 - lastS14).total_seconds() < 8:
            continue

        lastS14 = curS14

        # Match this trigger to an expLog entry by timestamp proximity
        for jj in range(len(eL)):
            eegTrig       = ii["orig_time"] + timedelta(seconds=np.round(ii["onset"]))
            qualisysTrig  = eL["RS"][jj]
            dt            = getTimeDiff(qualisysTrig, eegTrig)

            # Accept match if timestamps are within 5 seconds
            if dt.seconds >= 5:
                continue

            # Determine which leg was used from the direction cue
            # Left cue → right leg, Right cue → left leg
            used_leg = "right" if eL["Arrow_Direction"][jj] == "Left" else "left"

            # RS annotation at trigger onset
            ons.append(ii["onset"])
            dur.append(0.001)
            descr.append(f"RS_{used_leg}")

            # IC annotation at RS onset + ATR (anticipatory time to response)
            ons.append(ii["onset"] + eL["ATR"][jj])
            dur.append(0.001)
            descr.append(f"IC_{used_leg}")

    dataAnno = data.set_annotations(mne.Annotations(ons, dur, descr))
    return dataAnno


In [10]:
# -------------------------------------------------------------------------
# LOAD AND ANNOTATE EEG FILES
# For each EEG file:
#   1. Load raw BrainVision data
#   2. Find the corresponding expLog from derivatives/01_annot/
#   3. Set RS and IC annotations
#   4. Save annotated .fif to derivatives/01_annot/sub-XX/ses-XX/eeg/
#   5. Export events.tsv alongside the .fif (BIDS convention)
#
# Channel types and montage are set in the preprocessing step, not here.
# -------------------------------------------------------------------------
DERIVATIVES_PATH = wd / "derivatives" / "01_annot"

eegAnno = []

for eeg_file in eD:
    eeg_file = Path(eeg_file)

    # Derive subject and session from BIDS path (eeg/ → ses-XX → sub-XX)
    ses_name = eeg_file.parent.parent.name
    sub_name = eeg_file.parent.parent.parent.name

    print(f"Processing : {sub_name} / {ses_name} / {eeg_file.name}")

    # Locate corresponding expLog in derivatives/01_annot/
    log_path  = DERIVATIVES_PATH / sub_name / ses_name / "beh"
    log_files = sorted(log_path.glob("*_log.xlsx")) if log_path.exists() else []

    if len(log_files) == 0:
        print(f"  No expLog found, skipping : {sub_name} / {ses_name}")
        continue
    if len(log_files) > 1:
        print(f"  Multiple expLogs found, skipping : {sub_name} / {ses_name} — manual check needed")
        continue

    try:
        # Load raw EEG data (no channel types or montage set here)
        dataOUT = mne.io.read_raw_brainvision(eeg_file, preload=True)

        # Match EEG triggers to expLog and set RS/IC annotations
        dataAnno = setAnnotations(dataOUT, log_files[0])
        eegAnno.append(dataAnno)

        # -------------------------------------------------------------------------
        # SAVE ANNOTATED EEG FILE
        # Output: derivatives/01_annot/sub-XX/ses-XX/eeg/sub-XX_ses-XX_eeg.fif
        # -------------------------------------------------------------------------
        output_dir = DERIVATIVES_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)
        output_name = eeg_file.stem.replace("_eeg", "_desc-annot_eeg") + ".fif"
        output_path = output_dir / output_name       

        dataAnno.save(output_path, overwrite=True)
        print(f"  Saved EEG  : {output_path.name}")

        # -------------------------------------------------------------------------
        # EXPORT EVENTS.TSV (BIDS convention)
        # Columns: onset (s), duration (s), trial_type (RS/IC + leg)
        # One file per run, saved alongside the .fif.
        # -------------------------------------------------------------------------
        events_df = pd.DataFrame({
            "onset"      : dataAnno.annotations.onset,
            "duration"   : dataAnno.annotations.duration,
            "trial_type" : dataAnno.annotations.description,
        })

        events_path = output_dir / (eeg_file.stem.replace("_eeg", "_desc-annot_events") + ".tsv")
        events_df.to_csv(events_path, sep="\t", index=False)
        print(f"  Saved events : {events_path.name}")

    except Exception as e:
        print(f"  Error : {sub_name} / {ses_name} / {eeg_file.name} : {e}")

Processing : sub-01 / ses-01 / sub-01_ses-01_task-sidecut_run-01_eeg.vhdr
  Saved EEG  : sub-01_ses-01_task-sidecut_run-01_desc-annot_eeg.fif
  Saved events : sub-01_ses-01_task-sidecut_run-01_desc-annot_events.tsv
Processing : sub-01 / ses-01 / sub-01_ses-01_task-sidecut_run-02_eeg.vhdr
  Saved EEG  : sub-01_ses-01_task-sidecut_run-02_desc-annot_eeg.fif
  Saved events : sub-01_ses-01_task-sidecut_run-02_desc-annot_events.tsv
Processing : sub-01 / ses-01 / sub-01_ses-01_task-sidecut_run-03_eeg.vhdr
  Saved EEG  : sub-01_ses-01_task-sidecut_run-03_desc-annot_eeg.fif
  Saved events : sub-01_ses-01_task-sidecut_run-03_desc-annot_events.tsv
Processing : sub-01 / ses-01 / sub-01_ses-01_task-sidecut_run-04_eeg.vhdr
  Saved EEG  : sub-01_ses-01_task-sidecut_run-04_desc-annot_eeg.fif
  Saved events : sub-01_ses-01_task-sidecut_run-04_desc-annot_events.tsv
Processing : sub-02 / ses-01 / sub-02_ses-01_task-sidecut_run-01_eeg.vhdr
  Saved EEG  : sub-02_ses-01_task-sidecut_run-01_desc-annot_eeg.fi

## 4.2. Summary of annotations
Find number of events (annotations) that have been correctly processed. Also calculate the difference between # correct trials and # annotations.

In [11]:
# -------------------------------------------------------------------------
# ANNOTATION SUMMARY
# Counts RS and IC annotations per subject and session across all runs,
# merges with QTM trial counts for quality control, and saves to
# derivatives/01_annot/ as a session-specific Excel file.
# -------------------------------------------------------------------------

anno_records = []

for raw in eegAnno:
    fif_path = Path(raw.filenames[0])

    # Derive subject and session from BIDS path (eeg/ → ses-XX → sub-XX)
    ses_name = fif_path.parent.parent.name
    sub_name = fif_path.parent.parent.parent.name

    # Count IC annotations per leg (summed across all runs for this file)
    num_l = sum(1 for ann in raw.annotations if ann["description"] == "IC_left")
    num_r = sum(1 for ann in raw.annotations if ann["description"] == "IC_right")

    anno_records.append({
        "ID"          : sub_name,
        "Session"     : ses_name,
        "Annot Left"  : num_l,
        "Annot Right" : num_r,
        "Annot Total" : num_l + num_r,
    })

# -------------------------------------------------------------------------
# AGGREGATE ACROSS RUNS PER SUBJECT / SESSION
# Each run produces one entry — sum across runs within the same sub/ses.
# -------------------------------------------------------------------------
annot_df = (
    pd.DataFrame(anno_records)
    .groupby(["ID", "Session"], as_index=False)
    .sum()
)

# -------------------------------------------------------------------------
# MERGE WITH QTM TRIAL COUNTS AND COMPUTE DIFFERENCE
# sum_df is expected to be in scope from the QTM summary block above.
# -------------------------------------------------------------------------
merged_df = annot_df.merge(
    sum_df[["ID", "QTM Left total", "QTM Right total", "QTM total"]],
    on="ID",
    how="left"
)

merged_df["Diff Left"]  = merged_df["QTM Left total"]  - merged_df["Annot Left"]
merged_df["Diff Right"] = merged_df["QTM Right total"] - merged_df["Annot Right"]
merged_df["Diff Total"] = merged_df["QTM total"]       - merged_df["Annot Total"]

print(merged_df.to_string(index=False))

# -------------------------------------------------------------------------
# SAVE TO SESSION-SPECIFIC EXCEL FILE
# Appended as a second sheet to the existing trialslog_summary file,
# or saved separately per session if multiple sessions are present.
# -------------------------------------------------------------------------
for ses_name, ses_df in merged_df.groupby("Session"):
    date = datetime.now().strftime("%Y%m%d")
    summary_path = DERIVATIVES_PATH / f"trialslogSummary_{ses_name}_{date}.xlsx"

    if summary_path.exists():
        # Append annotation sheet to existing summary file
        with pd.ExcelWriter(summary_path, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
            ses_df.to_excel(writer, sheet_name="Annotations", index=False)
        print(f"Appended annotations sheet : {summary_path.name}")
    else:
        # Write new file if summary doesn't exist yet
        with pd.ExcelWriter(summary_path, mode="w", engine="openpyxl") as writer:
            ses_df.to_excel(writer, sheet_name="Annotations", index=False)
        print(f"Saved annotations summary  : {summary_path.name}")

    ID Session  Annot Left  Annot Right  Annot Total  QTM Left total  QTM Right total  QTM total  Diff Left  Diff Right  Diff Total
sub-01  ses-01          58           65          123              58               65        123          0           0           0
sub-02  ses-01          42           51           93              63               72        135         21          21          42
Appended annotations sheet : trialslogSummary_ses-01_20260516.xlsx


**Difference between QTM Trials and EEG Annot**
- sub-02: 42 Annot missing (EEG not measured in block 4, first series (7 trials) in block 3 missing - check in paper log)
- sub-05: 18 Annot missing (paper log: Some triggers did not reach EEG in block 3 -> 18 first trials of block 3 are missing

# 5. Concatenate and add metadata
Concatenate all runs per session and make one eeg-file per session for further processing

In [3]:
# -------------------------------------------------------------------------
# CONCATENATE ANNOTATED RUNS PER SUBJECT AND SESSION
# Loads annotated .fif files from derivatives/annotations/ and concatenates
# all runs per subject/session. Output is saved to derivatives/concat-raw/
# following BIDS naming: sub-XX_ses-XX_task-sidecut_desc-concat_eeg.fif
# -------------------------------------------------------------------------
wd = Path.cwd()  # project root; code lives at the same level as subject folders
ANNOTATIONS_PATH  = wd / "derivatives" / "01_annot"
CONCAT_RAW_PATH   = wd / "derivatives" / "02_concat-raw"

# -------------------------------------------------------------------------
# DISCOVER ANNOTATED EEG FILES
# Group by subject and session — concatenation is done within each ses-XX.
# -------------------------------------------------------------------------
session_files = defaultdict(list)  # key: (sub_name, ses_name)

for fif_file in sorted(ANNOTATIONS_PATH.rglob("*desc-annot_eeg.fif")):
    ses_name = fif_file.parent.parent.name   # eeg/ → ses-XX
    sub_name = fif_file.parent.parent.parent.name  # ses-XX → sub-XX
    session_files[(sub_name, ses_name)].append(fif_file)

print(f"Found files for {len(session_files)} subject/session combinations")

# -------------------------------------------------------------------------
# LOAD, CONCATENATE, AND SAVE
# -------------------------------------------------------------------------
for (sub_name, ses_name), fif_list in sorted(session_files.items()):
    print(f"\nProcessing : {sub_name} / {ses_name} ({len(fif_list)} runs)")

    try:
        # Load all runs for this subject/session
        raw_list = [mne.io.read_raw_fif(f, preload=True) for f in sorted(fif_list)]

        # Concatenate runs in order
        raw_concat = mne.concatenate_raws(raw_list, preload=True)
        print(f"  Concatenated {len(raw_list)} runs — {raw_concat.times[-1]:.1f} s total")

        # Count annotations per type
        ann_counts = defaultdict(int)
        for ann in raw_concat.annotations:
            ann_counts[ann["description"]] += 1
        for desc, count in sorted(ann_counts.items()):
            print(f"  {desc:<20} : {count}")

        # -------------------------------------------------------------------------
        # SAVE CONCATENATED FILE
        # BIDS output: derivatives/concat-raw/sub-XX/ses-XX/eeg/
        #              sub-XX_ses-XX_task-sidecut_desc-concat_eeg.fif
        # -------------------------------------------------------------------------
        output_dir = CONCAT_RAW_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = f"{sub_name}_{ses_name}_task-sidecut_desc-concat_eeg.fif"
        output_path = output_dir / output_name

        raw_concat.save(output_path, overwrite=True)
        print(f"  Saved : {output_path.name}")

    except Exception as e:
        print(f"  Error : {sub_name} / {ses_name} : {e}")

Found files for 2 subject/session combinations

Processing : sub-01 / ses-01 (4 runs)
  Concatenated 4 runs — 5806.7 s total
  BAD boundary         : 3
  EDGE boundary        : 3
  IC_left              : 58
  IC_right             : 65
  RS_left              : 58
  RS_right             : 65
  Saved : sub-01_ses-01_task-sidecut_desc-concat_eeg.fif

Processing : sub-02 / ses-01 (3 runs)
  Concatenated 3 runs — 4630.0 s total
  BAD boundary         : 2
  EDGE boundary        : 2
  IC_left              : 42
  IC_right             : 51
  RS_left              : 42
  RS_right             : 51
  Saved : sub-02_ses-01_task-sidecut_desc-concat_eeg.fif


# Summary

The following preparation steps were completed to ready the raw data for preprocessing:

1. [Finding all EEG recordings and log files](#1-list-all-eeg-recordings-and-log-files) — 
EEG header files (`.vhdr`) and behavioral log files (`.txt`) were discovered across all subjects and sessions following the BIDS folder structure (`sub-XX/ses-XX/eeg|beh`).

2. [Correcting channel names in `.vhdr` files](#2-correct-the-channel-names-in-vhdr-file) — 
The "Iz" channel was incorrectly labeled as `FCz` and was corrected to `Iz` directly in the BrainVision header files prior to loading.

3. [Processing the experimental log files](#3-process-experimental-log) — 
Behavioral log entries were matched to force plate data via recording timestamps. Time of initial contact (TOI) was extracted from force plate data and the anticipatory time to response (ATR = TOI − RS) was computed per trial. Invalid and incorrect trials were excluded. Processed logs were saved as `.xlsx` to `derivatives/01_annot/sub-XX/ses-XX/beh/`.

4. [Setting annotations at direction cue and initial ground contact](#4-match-eeg-and-force-plate-data) — 
EEG triggers (`S14`) were matched to expLog entries via timestamp proximity. Two annotations were set per trial: `RS_<leg>` at the reaction stimulus onset and `IC_<leg>` at the moment of initial ground contact. Annotated files and corresponding `events.tsv` files were saved to `derivatives/01_annot/sub-XX/ses-XX/eeg/`.

5. [Concatenating block recordings](#5-concatenate-block-recordings) — 
Individual run recordings were concatenated per subject and session and saved to `derivatives/02_concat-raw/sub-XX/ses-XX/eeg/` as a single `.fif` file, ready for preprocessing.